# Fault Tolerance Simulator – Demo

This notebook walks through the simulator defined in `src/simulator.py` and reproduces key scenarios from the chaos-test matrix in `docs/design.md`.

---

In [ ]:
# Ensure repo root is on path when running from the notebooks/ directory
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.simulator import Cluster

## 1. Baseline run – no failures

Verify that a clean 8-node cluster completes 200 steps without any failures or recoveries.

In [ ]:
cluster = Cluster(num_nodes=8, fail_rate=0.0, checkpoint_interval=20, seed=0)
result = cluster.run(steps=200)
print(result)

## 2. CT-01 – Single-node crash (low fail rate)

Mimic scenario CT-01: inject a low (5 %) per-node failure probability and observe elastic recovery.

In [ ]:
cluster = Cluster(num_nodes=8, fail_rate=0.05, checkpoint_interval=10, allow_elastic=True, seed=42)
result = cluster.run(steps=200)
print(result)
print(f"Final world size: {result.final_world_size} / 8")
print(f"Checkpoints saved: {result.num_checkpoints}")

## 3. CT-02 – Multi-node crash (10 % of cluster)

Higher failure rate to stress-test elastic rescaling.

In [ ]:
cluster = Cluster(num_nodes=16, fail_rate=0.10, checkpoint_interval=10, allow_elastic=True, seed=7)
result = cluster.run(steps=200)
print(result)

## 4. CT-05 – Non-elastic mode (job stops on any failure)

Validate that when `allow_elastic=False` the simulation terminates as soon as the first node fails.

In [ ]:
cluster = Cluster(num_nodes=8, fail_rate=0.15, checkpoint_interval=5, allow_elastic=False, seed=99)
result = cluster.run(steps=200)
print(result)
print(f"Stopped early: {result.completed_steps < result.requested_steps}")

## 5. Sweep – failure rate vs. completed steps

Plot how completed steps degrade as failure probability increases.

In [ ]:
fail_rates = [0.0, 0.02, 0.05, 0.10, 0.20, 0.40]
completed = []
surviving = []

for fr in fail_rates:
    c = Cluster(num_nodes=8, fail_rate=fr, checkpoint_interval=10, allow_elastic=True, seed=0)
    r = c.run(steps=200)
    completed.append(r.completed_steps)
    surviving.append(r.final_world_size)

print("fail_rate | completed_steps | surviving_nodes")
print("-" * 47)
for fr, cs, sv in zip(fail_rates, completed, surviving):
    print(f"{fr:9.2f} | {cs:15d} | {sv:15d}")

## 6. Checkpoint inspection

Inspect the checkpoint history saved during a run.

In [ ]:
cluster = Cluster(num_nodes=4, fail_rate=0.08, checkpoint_interval=5, allow_elastic=True, seed=12)
cluster.run(steps=30)

print("step | alive_nodes")
for ckpt in cluster.checkpoints:
    print(f"{ckpt.global_step:4d} | {ckpt.alive_nodes}")